### Reading the results

| Status | Meaning | Action |
|---|---|---|
| `PASS` | Expectation held | None |
| `WARN` | Known issue, quantified and documented | Review, don't block |
| `FAIL` | Expectation broken | **Stop. Do not build Clean.** |

### Check types

| Type | Question it answers |
|---|---|
| `ROW_COUNT` | Did anything load? |
| `UNIQUE` | Is the grain what we think it is? |
| `NOT_NULL` | Are key columns present? |
| `REFERENTIAL` | Do foreign keys resolve? |
| `DOMAIN` | Are categories from the expected set? |
| `RANGE` | Are numbers physically sensible? |
| `FORMAT` | Will values survive the cast in Clean? |
| `CONSISTENCY` | Do two facts agree with each other? |
| `COMPLETENESS` | How much is missing? |

In [0]:
use catalog `ftw-week-07`; select * from `01-raw`.`dq_check_results` limit 100;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS `ftw-week-07`.`01-raw`.dq_check_results (
    executed_at TIMESTAMP,
    layer       STRING,
    dataset     STRING,
    check_name  STRING,
    check_type  STRING,
    status      STRING,
    fail_count  BIGINT,
    details     STRING
);

-- Clear this layer's results so the tally reflects one run only.
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results WHERE layer = 'raw';

num_affected_rows
20


In [0]:
%sql
-- Did anything load?
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'courses', 'row_count_not_empty', 'ROW_COUNT',
    CASE WHEN COUNT(*) = 0 THEN 'FAIL' ELSE 'PASS' END,
    CASE WHEN COUNT(*) = 0 THEN 1 ELSE 0 END,
    'Zero rows = silent failure.'
FROM `ftw-week-07`.`01-raw`.courses;

-- Duplicate parent keys fan out every child join.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'courses', 'unique_module_presentation', 'UNIQUE',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Duplicate PK fans out every downstream join.'
FROM (SELECT code_module, code_presentation FROM `ftw-week-07`.`01-raw`.courses
      GROUP BY code_module, code_presentation HAVING COUNT(*) > 1);

-- All three columns are structural.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'courses', 'not_null_all_columns', 'NOT_NULL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Nulls break joins and duration logic.'
FROM `ftw-week-07`.`01-raw`.courses
WHERE code_module IS NULL OR code_presentation IS NULL OR module_presentation_length IS NULL;

-- A course cannot last zero days.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'courses', 'range_length_positive', 'RANGE',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Non-positive duration = parsing error.'
FROM `ftw-week-07`.`01-raw`.courses WHERE module_presentation_length <= 0;

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'assessments', 'row_count_not_empty', 'ROW_COUNT',
    CASE WHEN COUNT(*) = 0 THEN 'FAIL' ELSE 'PASS' END,
    CASE WHEN COUNT(*) = 0 THEN 1 ELSE 0 END, 'Zero rows = silent failure.'
FROM `ftw-week-07`.`01-raw`.assessments;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'assessments', 'unique_id_assessment', 'UNIQUE',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Duplicate PK double-counts every submission.'
FROM (SELECT id_assessment FROM `ftw-week-07`.`01-raw`.assessments GROUP BY id_assessment HAVING COUNT(*) > 1);

-- 'date' excluded: legitimately absent on Exam rows.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'assessments', 'not_null_structural', 'NOT_NULL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'date excluded -- Exam rows legitimately lack one.'
FROM `ftw-week-07`.`01-raw`.assessments
WHERE id_assessment IS NULL OR code_module IS NULL OR code_presentation IS NULL
   OR assessment_type IS NULL OR weight IS NULL;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'assessments', 'fk_to_courses', 'REFERENTIAL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Assessment with no parent course presentation.'
FROM `ftw-week-07`.`01-raw`.assessments a
LEFT JOIN `ftw-week-07`.`01-raw`.courses c ON a.code_module = c.code_module
                       AND a.code_presentation = c.code_presentation
WHERE c.code_module IS NULL;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'assessments', 'domain_assessment_type', 'DOMAIN',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Expected TMA, CMA, or Exam.'
FROM `ftw-week-07`.`01-raw`.assessments
WHERE assessment_type IS NULL OR assessment_type NOT IN ('TMA', 'CMA', 'Exam');

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'assessments', 'range_weight_0_100', 'RANGE',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Weight is a percentage.'
FROM `ftw-week-07`.`01-raw`.assessments WHERE weight < 0 OR weight > 100;

-- Fails only on values that are present but unparseable. '?' is handled below.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'assessments', 'format_date_numeric', 'FORMAT',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Present-but-unparseable dates would silently null out on cast.'
FROM `ftw-week-07`.`01-raw`.assessments
WHERE date IS NOT NULL AND TRIM(date) NOT IN ('', '?') AND TRY_CAST(date AS INT) IS NULL;

-- Records how many are missing. Informational.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'assessments', 'completeness_date', 'COMPLETENESS',
    'PASS', COUNT(*), 'Assessments with no due date. Expected: Exam rows only.'
FROM `ftw-week-07`.`01-raw`.assessments WHERE date IS NULL OR TRIM(date) IN ('', '?');

-- The missing dates must all be Exams. A TMA/CMA without one is a real gap.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'assessments', 'missing_date_only_on_exams', 'CONSISTENCY',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Only Exam rows may lack a due date.'
FROM `ftw-week-07`.`01-raw`.assessments
WHERE (date IS NULL OR TRIM(date) IN ('', '?')) AND assessment_type <> 'Exam';

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'vle', 'row_count_not_empty', 'ROW_COUNT',
    CASE WHEN COUNT(*) = 0 THEN 'FAIL' ELSE 'PASS' END,
    CASE WHEN COUNT(*) = 0 THEN 1 ELSE 0 END, 'Zero rows = silent failure.'
FROM `ftw-week-07`.`01-raw`.vle;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'vle', 'unique_id_site', 'UNIQUE',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Duplicate PK double-counts clicks in student_vle.'
FROM (SELECT id_site FROM `ftw-week-07`.`01-raw`.vle GROUP BY id_site HAVING COUNT(*) > 1);

-- week_from/week_to excluded: legitimately absent for always-on resources.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'vle', 'not_null_structural', 'NOT_NULL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'week_from/week_to excluded -- absent for always-on resources.'
FROM `ftw-week-07`.`01-raw`.vle
WHERE id_site IS NULL OR code_module IS NULL OR code_presentation IS NULL OR activity_type IS NULL;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'vle', 'fk_to_courses', 'REFERENTIAL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Resource with no parent course presentation.'
FROM `ftw-week-07`.`01-raw`.vle v
LEFT JOIN `ftw-week-07`.`01-raw`.courses c ON v.code_module = c.code_module
                       AND v.code_presentation = c.code_presentation
WHERE c.code_module IS NULL;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'vle', 'completeness_week_range', 'COMPLETENESS',
    'PASS', COUNT(*), 'Resources with no week range. Expected for always-on resources.'
FROM `ftw-week-07`.`01-raw`.vle
WHERE week_from IS NULL OR TRIM(week_from) IN ('', '?')
   OR week_to   IS NULL OR TRIM(week_to)   IN ('', '?');

-- Both weeks must be absent together, or present together.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'vle', 'week_range_missing_as_pair', 'CONSISTENCY',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Half a range looks valid but is not.'
FROM `ftw-week-07`.`01-raw`.vle
WHERE (week_from IS NULL OR TRIM(week_from) IN ('', '?'))
   <> (week_to   IS NULL OR TRIM(week_to)   IN ('', '?'));

-- When both are present, the range must run forwards.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'vle', 'range_week_order', 'RANGE',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'week_from after week_to = column swap or parse error.'
FROM `ftw-week-07`.`01-raw`.vle
WHERE TRY_CAST(week_from AS INT) IS NOT NULL AND TRY_CAST(week_to AS INT) IS NOT NULL
  AND TRY_CAST(week_from AS INT) > TRY_CAST(week_to AS INT);

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_info', 'row_count_not_empty', 'ROW_COUNT',
    CASE WHEN COUNT(*) = 0 THEN 'FAIL' ELSE 'PASS' END,
    CASE WHEN COUNT(*) = 0 THEN 1 ELSE 0 END, 'Zero rows = silent failure.'
FROM `ftw-week-07`.`01-raw`.student_info;

-- Grain is the enrollment triplet, NOT id_student alone.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_info', 'unique_enrollment_grain', 'UNIQUE',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Duplicates = two conflicting outcomes for one enrollment.'
FROM (SELECT code_module, code_presentation, id_student FROM `ftw-week-07`.`01-raw`.student_info
      GROUP BY code_module, code_presentation, id_student HAVING COUNT(*) > 1);

-- imd_band excluded: documented as legitimately missing.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_info', 'not_null_structural', 'NOT_NULL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'imd_band excluded -- documented as missing on some rows.'
FROM `ftw-week-07`.`01-raw`.student_info
WHERE code_module IS NULL OR code_presentation IS NULL OR id_student IS NULL
   OR gender IS NULL OR final_result IS NULL;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_info', 'fk_to_courses', 'REFERENTIAL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Enrollment with no parent course presentation.'
FROM `ftw-week-07`.`01-raw`.student_info si
LEFT JOIN `ftw-week-07`.`01-raw`.courses c ON si.code_module = c.code_module
                       AND si.code_presentation = c.code_presentation
WHERE c.code_module IS NULL;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_info', 'domain_gender', 'DOMAIN',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*), 'Expected M or F.'
FROM `ftw-week-07`.`01-raw`.student_info WHERE gender IS NOT NULL AND gender NOT IN ('M', 'F');

-- final_result drives every dropout question -- drift here corrupts the dashboard.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_info', 'domain_final_result', 'DOMAIN',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Expected Withdrawn, Fail, Pass, or Distinction.'
FROM `ftw-week-07`.`01-raw`.student_info
WHERE final_result IS NOT NULL
  AND final_result NOT IN ('Withdrawn', 'Fail', 'Pass', 'Distinction');

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_info', 'domain_disability', 'DOMAIN',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*), 'Expected Y or N.'
FROM `ftw-week-07`.`01-raw`.student_info WHERE disability IS NOT NULL AND disability NOT IN ('Y', 'N');

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_info', 'range_attempts_credits', 'RANGE',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Negative attempts or non-positive credits are impossible.'
FROM `ftw-week-07`.`01-raw`.student_info WHERE num_of_prev_attempts < 0 OR studied_credits <= 0;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_info', 'completeness_imd_band', 'COMPLETENESS',
    'PASS', COUNT(*),
    'Enrollments with no IMD band. Clean must convert to NULL or "?" becomes a category.'
FROM `ftw-week-07`.`01-raw`.student_info WHERE imd_band IS NULL OR TRIM(imd_band) IN ('', '?');

-- Categorical columns are never cast, so nothing else would catch a sentinel here.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_info', 'sentinel_in_required_categoricals', 'DOMAIN',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'These columns should never carry a missing-value marker.'
FROM `ftw-week-07`.`01-raw`.student_info
WHERE TRIM(gender) IN ('', '?') OR TRIM(region) IN ('', '?')
   OR TRIM(age_band) IN ('', '?') OR TRIM(highest_education) IN ('', '?')
   OR TRIM(disability) IN ('', '?') OR TRIM(final_result) IN ('', '?');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_registration', 'row_count_not_empty', 'ROW_COUNT',
    CASE WHEN COUNT(*) = 0 THEN 'FAIL' ELSE 'PASS' END,
    CASE WHEN COUNT(*) = 0 THEN 1 ELSE 0 END, 'Zero rows = silent failure.'
FROM `ftw-week-07`.`01-raw`.student_registration;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_registration', 'unique_enrollment_grain', 'UNIQUE',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Duplicates = two conflicting registrations for one enrollment.'
FROM (SELECT code_module, code_presentation, id_student FROM `ftw-week-07`.`01-raw`.student_registration
      GROUP BY code_module, code_presentation, id_student HAVING COUNT(*) > 1);

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_registration', 'not_null_keys', 'NOT_NULL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Key columns only. Both date columns handled separately.'
FROM `ftw-week-07`.`01-raw`.student_registration
WHERE code_module IS NULL OR code_presentation IS NULL OR id_student IS NULL;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_registration', 'fk_to_student_info', 'REFERENTIAL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Registration with no matching enrollment.'
FROM `ftw-week-07`.`01-raw`.student_registration sr
LEFT JOIN `ftw-week-07`.`01-raw`.student_info si ON sr.code_module = si.code_module
                            AND sr.code_presentation = si.code_presentation
                            AND sr.id_student = si.id_student
WHERE si.id_student IS NULL;

-- Present-but-unparseable only. Negative values are valid (registered before start).
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_registration', 'format_dates_numeric', 'FORMAT',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Negative values are valid (early registration) and not flagged.'
FROM `ftw-week-07`.`01-raw`.student_registration
WHERE (date_registration   IS NOT NULL AND TRIM(date_registration)   NOT IN ('', '?')
       AND TRY_CAST(date_registration   AS INT) IS NULL)
   OR (date_unregistration IS NOT NULL AND TRIM(date_unregistration) NOT IN ('', '?')
       AND TRY_CAST(date_unregistration AS INT) IS NULL);

-- Real gap. WARN so it stays visible without permanently blocking the gate.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_registration', 'completeness_date_registration', 'COMPLETENESS',
    CASE WHEN COUNT(*) = 0 THEN 'PASS'
         WHEN COUNT(*) <= 50 THEN 'WARN'
         ELSE 'FAIL' END,
    COUNT(*),
    'Expected 0, found 45. Known gap -- see decisions.md. Clean stores NULL, not 0. FAILs above 50.'
FROM `ftw-week-07`.`01-raw`.student_registration
WHERE date_registration IS NULL OR TRIM(date_registration) IN ('', '?');

-- Not a gap -- this is the non-withdrawal population.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_registration', 'completeness_withdrawals', 'COMPLETENESS',
    'PASS', COUNT(*),
    'Enrollments that never unregistered. Cross-check against final_result <> Withdrawn.'
FROM `ftw-week-07`.`01-raw`.student_registration
WHERE date_unregistration IS NULL OR TRIM(date_unregistration) IN ('', '?');

-- No derivable timeline at all -- Mart must exclude these from duration metrics.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_registration', 'completeness_both_dates_missing', 'COMPLETENESS',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'WARN' END, COUNT(*),
    'Neither date present. Flag in Clean so Mart excludes them from duration metrics.'
FROM `ftw-week-07`.`01-raw`.student_registration
WHERE (date_registration   IS NULL OR TRIM(date_registration)   IN ('', '?'))
  AND (date_unregistration IS NULL OR TRIM(date_unregistration) IN ('', '?'));

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_assessment', 'row_count_not_empty', 'ROW_COUNT',
    CASE WHEN COUNT(*) = 0 THEN 'FAIL' ELSE 'PASS' END,
    CASE WHEN COUNT(*) = 0 THEN 1 ELSE 0 END, 'Zero rows = silent failure.'
FROM `ftw-week-07`.`01-raw`.student_assessment;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_assessment', 'unique_submission_grain', 'UNIQUE',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Duplicates = two conflicting scores for one submission.'
FROM (SELECT id_assessment, id_student FROM `ftw-week-07`.`01-raw`.student_assessment
      GROUP BY id_assessment, id_student HAVING COUNT(*) > 1);

-- score excluded: absent for non-submissions.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_assessment', 'not_null_structural', 'NOT_NULL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'score excluded -- absent for non-submissions.'
FROM `ftw-week-07`.`01-raw`.student_assessment
WHERE id_assessment IS NULL OR id_student IS NULL OR date_submitted IS NULL;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_assessment', 'fk_to_assessments', 'REFERENTIAL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Submission against a non-existent assessment.'
FROM `ftw-week-07`.`01-raw`.student_assessment sa
LEFT JOIN `ftw-week-07`.`01-raw`.assessments a ON sa.id_assessment = a.id_assessment
WHERE a.id_assessment IS NULL;

-- This table has no module/presentation columns, so joins on id_student alone.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_assessment', 'fk_to_student', 'REFERENTIAL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Submission from a student with no enrollment record anywhere.'
FROM (SELECT DISTINCT sa.id_student FROM `ftw-week-07`.`01-raw`.student_assessment sa
      LEFT ANTI JOIN `ftw-week-07`.`01-raw`.student_info si ON sa.id_student = si.id_student);

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_assessment', 'domain_is_banked', 'DOMAIN',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*), 'Expected 0 or 1.'
FROM `ftw-week-07`.`01-raw`.student_assessment WHERE is_banked NOT IN (0, 1);

-- Present-but-invalid scores only.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_assessment', 'format_range_score', 'FORMAT',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Present scores must parse as a number 0-100.'
FROM `ftw-week-07`.`01-raw`.student_assessment
WHERE score IS NOT NULL AND TRIM(score) NOT IN ('', '?')
  AND (TRY_CAST(score AS DOUBLE) IS NULL
       OR TRY_CAST(score AS DOUBLE) < 0 OR TRY_CAST(score AS DOUBLE) > 100);

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_assessment', 'completeness_score', 'COMPLETENESS',
    'PASS', COUNT(*),
    'Submissions with no score. Clean must use NULL, not 0 -- these are different facts.'
FROM `ftw-week-07`.`01-raw`.student_assessment WHERE score IS NULL OR TRIM(score) IN ('', '?');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_vle', 'row_count_not_empty', 'ROW_COUNT',
    CASE WHEN COUNT(*) = 0 THEN 'FAIL' ELSE 'PASS' END,
    CASE WHEN COUNT(*) = 0 THEN 1 ELSE 0 END, 'Zero rows = silent failure.'
FROM `ftw-week-07`.`01-raw`.student_vle;

-- No UNIQUE check: duplicates on the daily key are EXPECTED at Raw.
-- What matters is that every column Clean will GROUP BY is present.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_vle', 'aggregation_key_complete', 'NOT_NULL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Raw is per-session; Clean aggregates. The GROUP BY key must be complete.'
FROM `ftw-week-07`.`01-raw`.student_vle
WHERE code_module IS NULL OR code_presentation IS NULL OR id_student IS NULL
   OR id_site IS NULL OR date IS NULL OR sum_click IS NULL;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_vle', 'fk_to_vle', 'REFERENTIAL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Interaction with a non-existent VLE resource.'
FROM `ftw-week-07`.`01-raw`.student_vle sv
LEFT JOIN `ftw-week-07`.`01-raw`.vle v ON sv.id_site = v.id_site
WHERE v.id_site IS NULL;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_vle', 'fk_to_student_info', 'REFERENTIAL',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Engagement data for an enrollment that does not exist.'
FROM `ftw-week-07`.`01-raw`.student_vle sv
LEFT JOIN `ftw-week-07`.`01-raw`.student_info si ON sv.code_module = si.code_module
                            AND sv.code_presentation = si.code_presentation
                            AND sv.id_student = si.id_student
WHERE si.id_student IS NULL;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_vle', 'range_sum_click_positive', 'RANGE',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'A logged interaction implies at least one click.'
FROM `ftw-week-07`.`01-raw`.student_vle WHERE sum_click <= 0;

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_vle', 'format_date_numeric', 'FORMAT',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END, COUNT(*),
    'Present-but-unparseable dates would silently null out on cast.'
FROM `ftw-week-07`.`01-raw`.student_vle
WHERE date IS NOT NULL AND TRIM(date) NOT IN ('', '?') AND TRY_CAST(date AS INT) IS NULL;

-- Baseline for Clean to reconcile against. Rows change; this number must not.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_vle', 'reconciliation_total_clicks', 'COMPLETENESS',
    'PASS', SUM(sum_click),
    'Total clicks at Raw. Clean must reproduce this exactly after aggregating.'
FROM `ftw-week-07`.`01-raw`.student_vle;

-- Early warning for future batches. Baseline 1.2596 rows per daily key.
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'student_vle', 'fanout_ratio_stable', 'RANGE',
    CASE WHEN ratio BETWEEN 1.0 AND 2.0 THEN 'PASS' ELSE 'FAIL' END,
    CAST(ROUND(ratio * 10000) AS BIGINT),
    CONCAT('Rows per daily key x10000 (baseline 12596). Drift means the source grain changed. Now: ',
           CAST(ROUND(ratio, 4) AS STRING))
FROM (SELECT COUNT(*) / COUNT(DISTINCT CONCAT_WS('|', code_module, code_presentation,
                                                 id_student, id_site, date)) AS ratio
      FROM `ftw-week-07`.`01-raw`.student_vle);

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT CURRENT_TIMESTAMP(), 'raw', 'cross_table', 'withdrawal_flag_agrees_with_date', 'CONSISTENCY',
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'WARN' END, COUNT(*),
    'final_result and unregistration date disagree. Understand this count before Mart uses either for dropout analysis.'
FROM `ftw-week-07`.`01-raw`.student_info si
JOIN `ftw-week-07`.`01-raw`.student_registration sr ON si.code_module = sr.code_module
                               AND si.code_presentation = sr.code_presentation
                               AND si.id_student = sr.id_student
WHERE (si.final_result = 'Withdrawn')
   <> (sr.date_unregistration IS NOT NULL AND TRIM(sr.date_unregistration) NOT IN ('', '?'));

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
SELECT
    dataset,
    COUNT(*)                                         AS checks,
    SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) AS passed,
    SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) AS warnings,
    SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS failed
FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE layer = 'raw'
GROUP BY dataset
ORDER BY failed DESC, dataset;

dataset,checks,passed,warnings,failed
assessments,9,9,0,0
courses,4,4,0,0
cross_table,1,0,1,0
student_assessment,8,8,0,0
student_info,10,10,0,0
student_registration,8,6,2,0
student_vle,8,8,0,0
vle,7,7,0,0


In [0]:
%sql
SELECT dataset, check_type, check_name, fail_count, details
FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE layer = 'raw' AND status = 'FAIL'
ORDER BY dataset, check_type;

dataset,check_type,check_name,fail_count,details


In [0]:
%sql
--known issues
SELECT dataset, check_name, fail_count AS affected_rows, details
FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE layer = 'raw' AND status = 'WARN'
ORDER BY affected_rows DESC;

dataset,check_name,affected_rows,details
cross_table,withdrawal_flag_agrees_with_date,102,final_result and unregistration date disagree. Understand this count before Mart uses either for dropout analysis.
student_registration,completeness_date_registration,45,"Expected 0, found 45. Known gap -- see decisions.md. Clean stores NULL, not 0. FAILs above 50."
student_registration,completeness_both_dates_missing,6,Neither date present. Flag in Clean so Mart excludes them from duration metrics.


In [0]:
%sql
--missing value inventory for cleaning
SELECT dataset, check_name, fail_count AS affected_rows, details
FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE layer = 'raw' AND check_type = 'COMPLETENESS'
ORDER BY affected_rows DESC;

dataset,check_name,affected_rows,details
student_vle,reconciliation_total_clicks,39605099,Total clicks at Raw. Clean must reproduce this exactly after aggregating.
student_registration,completeness_withdrawals,22521,Enrollments that never unregistered. Cross-check against final_result <> Withdrawn.
vle,completeness_week_range,5243,Resources with no week range. Expected for always-on resources.
student_info,completeness_imd_band,1111,"Enrollments with no IMD band. Clean must convert to NULL or ""?"" becomes a category."
student_assessment,completeness_score,173,"Submissions with no score. Clean must use NULL, not 0 -- these are different facts."
student_registration,completeness_date_registration,45,"Expected 0, found 45. Known gap -- see decisions.md. Clean stores NULL, not 0. FAILs above 50."
assessments,completeness_date,11,Assessments with no due date. Expected: Exam rows only.
student_registration,completeness_both_dates_missing,6,Neither date present. Flag in Clean so Mart excludes them from duration metrics.
